# Gold — Dinámica de carrera por caballo (`gold_derby_race_dynamics`)

## Propuesta

**Modelo propuesto:** un modelo **no supervisado** que agrupa a cada caballo-carrera según su
**estilo de carrera** (KMeans sobre el perfil de velocidad) y detecta **carreras atípicas**
a partir de la distancia de cada caballo al centroide de su grupo.

**Diferencia con el clasificador top-3:** el modelo existente es **supervisado y predictivo**:
estima, antes de la largada, la probabilidad de que un caballo termine entre los tres primeros.
El modelo propuesto no predice un resultado. **Describe** cómo corrió el caballo una vez
terminada la carrera: si arrancó fuerte y se apagó, si cerró con fuerza, si mantuvo un ritmo
parejo o si tuvo frenazos bruscos.

**Por qué puede usar el GPS sin incurrir en leakage:** el *data leakage* ocurre cuando un modelo
usa información que no estaría disponible en el momento en que se va a usar la predicción.
El clasificador top-3 se usa **antes** de la carrera, y en ese momento la velocidad GPS todavía
no existe, por eso la excluye. Este modelo se usa **después** de la carrera, para caracterizar
lo que ya ocurrió. En ese momento el tracking completo ya está disponible, así que usarlo no
filtra información del futuro: es justamente el dato que queremos describir.

**Esta tabla** resume, para cada caballo-carrera, la **forma** de su perfil de velocidad en el
tiempo (velocidad por cuartos, capacidad de arranque y de cierre, regularidad del ritmo y
desaceleraciones bruscas), información que `gold_derby_ml_features` no captura porque solo
agrega la magnitud de la velocidad (promedio, máximo, desviación).

In [0]:
%run ../00-setup/config

# Configuración centralizada del pipeline Big Data Derby 2022

Este notebook se importa con `%run ../00-setup/config`.
Centraliza paths, schemas, nombres de tabla y parámetros del pipeline.

## Arquitectura real del curso

- **Catálogo compartido**: `20080_databricks`, ya creado por el profesor.
  El estudiante NO crea catálogos ni schemas: solo necesita `USE CATALOG`
  sobre el catálogo y ser dueño de su schema, que también crea el profesor.
  El nombre del catálogo cambia entre iteraciones del curso: confírmalo con
  tu docente antes de correr nada. Como empieza por dígito, en SQL siempre
  va entre backticks.
- **Un schema por estudiante**: `<CATALOG>.<SCHEMA>`, donde `SCHEMA` es tu
  `nombre_apellido` en snake_case (por ejemplo `camilo_soto`). Es lo ÚNICO
  que debes editar en este notebook.
- **Landing EN TU PROPIO Volume**: a diferencia del módulo 06 (que lee de un
  landing compartido de solo lectura), aquí **cada estudiante descarga el
  dataset de Kaggle a su propio Volume** `datos`, bajo `landing/derby`. Lo
  hace el notebook `00-setup/02_download_dataset.py`.
- **Capas medallion como tablas administradas de Unity Catalog**: bronze,
  silver y gold se escriben con `saveAsTable` en `<CATALOG>.<SCHEMA>`. UC
  gestiona el almacenamiento, así que las tablas se consultan por nombre
  desde SQL, aparecen en Catalog Explorer con linaje y admiten `GRANT` por
  tabla. Las capas se distinguen por prefijo en el nombre (`bronze_derby_`,
  `silver_derby_`, `gold_derby_`).

Catalog:            51810_databricks
Schema:             sebastian_velasco
Volume:             landing
Landing (tuyo):     /Volumes/51810_databricks/sebastian_velasco/landing/landing/derby
Kaggle competition: big-data-derby-2022

Bronze race:        `51810_databricks`.`sebastian_velasco`.bronze_derby_race
Bronze start:       `51810_databricks`.`sebastian_velasco`.bronze_derby_start
Bronze tracking:    `51810_databricks`.`sebastian_velasco`.bronze_derby_tracking
Silver races:       `51810_databricks`.`sebastian_velasco`.silver_derby_races
Silver starts:      `51810_databricks`.`sebastian_velasco`.silver_derby_starts
Silver tracking:    `51810_databricks`.`sebastian_velasco`.silver_derby_tracking
Gold race_summary:  `51810_databricks`.`sebastian_velasco`.gold_derby_race_summary
Gold track_metrics: `51810_databricks`.`sebastian_velasco`.gold_derby_track_metrics
Gold jockey_perf:   `51810_databricks`.`sebastian_velasco`.gold_derby_jockey_performance
Gold ml_features:   `51810_databricks`.`

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

tracking = spark.table(T_SILVER_TRACKING)
starts = spark.table(T_SILVER_STARTS)

print("Frames en tracking:", tracking.count())
print("Caballo-carrera en starts:", starts.count())
print("START_KEYS:", START_KEYS)
print("TRACKING_HZ:", TRACKING_HZ)
print("Tabla destino:", T_GOLD_RACE_DYNAMICS)

Frames en tracking: 5228430
Caballo-carrera en starts: 14914
START_KEYS: ['track_id', 'race_date', 'race_number', 'program_number']
TRACKING_HZ: 4.0
Tabla destino: `51810_databricks`.`sebastian_velasco`.gold_derby_race_dynamics


In [0]:
# Exploración sobre los datos de Silver (antes de cualquier limpieza)
raw = tracking.filter(F.col("speed_kmh").isNotNull())

frames_per_horse = raw.groupBy(*START_KEYS).agg(
    F.count("*").alias("n_frames"),
    F.min("trakus_index").alias("idx_min"),
    F.max("trakus_index").alias("idx_max"),
)

print("Caballo-carrera con tracking:", frames_per_horse.count())
frames_per_horse.select("n_frames", "idx_min", "idx_max").summary(
    "min", "1%", "5%", "25%", "50%", "75%", "max"
).display()

Caballo-carrera con tracking: 14915


summary,n_frames,idx_min,idx_max
min,3,2,4
1%,246,2,247
5%,263,2,264
25%,293,2,294
50%,335,2,336
75%,395,2,396
max,1061,4,1062


In [0]:
for umbral in [20, 50, 100, 150, 200]:
    n = frames_per_horse.filter(F.col("n_frames") < umbral).count()
    print(f"Caballos con menos de {umbral} frames: {n}")

# Mira los casos más cortos para entender qué son
frames_per_horse.orderBy("n_frames").limit(10).display()

Caballos con menos de 20 frames: 5
Caballos con menos de 50 frames: 11
Caballos con menos de 100 frames: 15
Caballos con menos de 150 frames: 25
Caballos con menos de 200 frames: 29


track_id,race_date,race_number,program_number,n_frames,idx_min,idx_max
BEL,2019-07-06,3,3,3,2,4
AQU,2019-12-06,3,1,15,2,16
AQU,2019-01-06,4,1,16,2,17
BEL,2019-04-27,6,3,17,2,18
AQU,2019-11-20,4,3,17,2,18
AQU,2019-12-28,7,7,21,2,22
AQU,2019-11-02,9,4,28,2,29
AQU,2019-11-02,9,3,36,2,37
BEL,2019-05-17,2,4,42,2,43
AQU,2019-12-21,8,4,45,2,46


In [0]:
# Ruido GPS en los datos CRUDOS de Silver (antes de limpiar)
total_frames = tracking.filter(F.col("speed_kmh").isNotNull()).count()

for limite in [80, 90, 100, 150, 500]:
    n = tracking.filter(F.col("speed_kmh") > limite).count()
    print(f"Frames con speed_kmh > {limite}: {n:>8,}  ({n / total_frames:.4%})")

# ¿Cuántos caballos tienen al menos un frame imposible?
n_caballos = (tracking
              .filter(F.col("speed_kmh") > 90)
              .select(*START_KEYS).distinct().count())
print(f"\nCaballo-carrera con algún frame > 90 km/h: {n_caballos}")

Frames con speed_kmh > 80:   50,205  (0.9630%)
Frames con speed_kmh > 90:    1,588  (0.0305%)
Frames con speed_kmh > 100:      808  (0.0155%)
Frames con speed_kmh > 150:      348  (0.0067%)
Frames con speed_kmh > 500:       66  (0.0013%)

Caballo-carrera con algún frame > 90 km/h: 882


### Limpieza de velocidades imposibles (ruido GPS)

La exploración del cambio de velocidad entre frames consecutivos mostró caídas de hasta
**−2.305 km/h en 0,25 s**, físicamente imposibles. Provienen de frames donde la posición GPS
"salta": la distancia Haversine resulta enorme y la velocidad calculada en Silver supera
los miles de km/h. Estos frames contaminaban los promedios por cuarto (máximos de ~97 km/h
promedio en un cuarto completo) y el `speed_cv` (máximo de 2,44).

| Velocidad | Frames | % del total |
|---|---|---|
| > 80 km/h | 50.204 | 0,96% |
| > 90 km/h | 1.588 | 0,03% |
| > 150 km/h | 348 | 0,007% |
| > 500 km/h | 66 | 0,001% |

**Decisión:** se eliminan los frames con `speed_kmh > 90` (`MAX_SPEED_KMH = 90`).

- La velocidad máxima de un caballo de carreras ronda los 70 km/h. Entre 80 y 90 km/h hay
  ~50 mil frames que corresponden al error normal de medición del GPS (1–2 m por frame a 4 Hz).
  Por encima de 90 el conteo cae 30 veces, lo que marca el paso de ruido de medición a saltos.
- Se eliminan **frames**, no caballos: los 1.588 frames malos están repartidos en **882
  caballo-carrera (~6%)**. Descartar caballos completos perdería esas carreras por un error de
  menos de un segundo; descartar frames pierde solo el 0,03% de los datos.
- Se descartó recortar la velocidad a 90 (*capping*), porque introduce un valor inventado y
  seguiría generando un frenazo falso al compararlo con el frame siguiente.
- Al eliminar un frame, el `lag` del frame siguiente se compara con el anterior al eliminado,
  con 0,5 s de diferencia en lugar de 0,25 s. Dado que afecta al 0,03% de los frames,
  el efecto es despreciable.
- **No se filtran las velocidades bajas.** Un caballo que se detiene a mitad de carrera
  (*pulled up*) es un comportamiento real que el detector de anomalías debe poder encontrar.

### Recorte de la espera en el partidor

El tracking comienza **antes de la largada**: 1.444 caballo-carrera (~10%) tienen sus primeros
frames con el caballo detenido en el partidor (`speed_kmh < 5`), unos 3 frames en promedio.
Aunque es menos de un segundo, esos frames en 0 bajan la velocidad media del Q1 en 2–3 km/h
y desplazan el `early_speed_ratio` en ~0,03–0,04, del mismo orden que la variación real de
esa feature entre caballos (1,00–1,07). Sin recortarlos, un caballo parecería de arranque
lento solo porque la grabación empezó antes.

**Decisión:** la carrera de cada caballo empieza en el primer frame donde su velocidad es
≥ 5 km/h **de forma sostenida durante 1 s** (ese frame y los 3 siguientes, `rowsBetween(0, 3)`).
Exigir 1 s evita que un pico aislado de ruido GPS sobre un caballo quieto se confunda con el
arranque. Se descartan los frames anteriores.

**No se recortan los frames quietos del final:** solo 56 caballos terminan detenidos, con
~5 s de detención cada uno. Corresponden a caballos que se detuvieron o fueron retirados
(*pulled up*), un comportamiento real que el detector de anomalías debe poder encontrar.

Resultado: los caballos con el primer frame quieto pasan de 1.444 a 0, y los frames quietos
en Q1 de 4.518 a 31.

In [ ]:
MAX_SPEED_KMH = 90   # velocidad física máxima plausible; lo que pase de aquí es ruido GPS
STOP_KMH = 5         # por debajo de esto el caballo está quieto

horse_ordered_w = Window.partitionBy(*START_KEYS).orderBy("trakus_index")
horse_w = Window.partitionBy(*START_KEYS)

valid = (
    tracking
    # 1. Quitar el primer frame (NULL) y el ruido GPS
    .filter(F.col("speed_kmh").isNotNull() & (F.col("speed_kmh") <= MAX_SPEED_KMH))
    # 2. Velocidad mínima en este frame y los 3 siguientes (1 segundo)
    .withColumn("min_next_1s", F.min("speed_kmh").over(horse_ordered_w.rowsBetween(0, 3)))
    # 3. Primer frame donde el caballo se mueve de forma sostenida = arranque real
    .withColumn(
        "start_idx",
        F.min(F.when(F.col("min_next_1s") >= STOP_KMH, F.col("trakus_index"))).over(horse_w)
    )
    # 4. Descartar todo lo anterior al arranque
    .filter(F.col("trakus_index") >= F.col("start_idx"))
    .drop("min_next_1s", "start_idx")
)


### Filtro de frames mínimos

Cada frame es una medición GPS del caballo (4 por segundo). La exploración mostró que la mediana
es de **335 frames por caballo-carrera** (unos 84 s de carrera), pero existe una cola de registros
cortados: el mínimo es de **3 frames** (menos de 1 s de datos).

Se descartan los caballo-carrera con **menos de 100 frames** (`MIN_FRAMES = 100`):

- Elimina **15 de 14.915** caballo-carrera (≈0,1%), repartidos en carreras distintas, por lo que
  corresponden a fallas puntuales del tracking y no a carreras completas.
- Ninguna carrera real dura menos de 25 s, así que estos casos son registros incompletos,
  no carreras cortas.
- Con 100 frames, cada cuarto de carrera tiene al menos **25 frames (~6 s)**, suficiente para que
  el promedio de velocidad por cuarto no dependa de uno o dos frames con ruido GPS.

Los caballo-carrera entre 100 y 200 frames (14 casos) se conservan: si su perfil es atípico,
el detector de anomalías de la Parte 2 debería marcarlos.

**Hallazgo adicional:** en todos los caballos `n_frames = idx_max − idx_min + 1`, es decir,
**no hay huecos en el tracking**. Por eso la partición en cuartos con `floor` (avance de carrera)
y con `ntile` (cantidad de frames) darían resultados prácticamente idénticos. Se usa `floor`
porque representa de forma más directa la idea de "cuartos del recorrido".

In [0]:
MIN_FRAMES = 100

horse_w = Window.partitionBy(*START_KEYS)   # todos los frames de un caballo-carrera

base = (
    valid
    .withColumn("n_frames", F.count("*").over(horse_w))
    .withColumn("idx_min",  F.min("trakus_index").over(horse_w))
    .withColumn("idx_max",  F.max("trakus_index").over(horse_w))
    .filter(F.col("n_frames") >= MIN_FRAMES)
)

print("Frames en base:", base.count())
print("Caballo-carrera en base:", base.select(*START_KEYS).distinct().count())
base.limit(5).display()

Frames en base: 5206236
Caballo-carrera en base: 14900


track_id,race_date,race_number,program_number,trakus_index,latitude,longitude,_source_file,_ingested_at,step_distance_m,dt_seconds,speed_mps,speed_kmh,_silver_at,n_frames,idx_min,idx_max
AQU,2019-01-01,1,1,2,40.6694051526183,-73.8292030605591,dbfs:/Volumes/51810_databricks/sebastian_velasco/landing/landing/derby/nyra_tracking_table.csv,2026-10-17T02:03:43.918Z,0.529,0.25,2.116,7.62,2026-10-17T02:13:40.632Z,315,2,316
AQU,2019-01-01,1,1,3,40.6694113624261,-73.8292000780746,dbfs:/Volumes/51810_databricks/sebastian_velasco/landing/landing/derby/nyra_tracking_table.csv,2026-10-17T02:03:43.918Z,0.735,0.25,2.94,10.58,2026-10-17T02:13:40.632Z,315,2,316
AQU,2019-01-01,1,1,4,40.6694207810975,-73.8291957066611,dbfs:/Volumes/51810_databricks/sebastian_velasco/landing/landing/derby/nyra_tracking_table.csv,2026-10-17T02:03:43.918Z,1.11,0.25,4.44,15.98,2026-10-17T02:13:40.632Z,315,2,316
AQU,2019-01-01,1,1,5,40.6694325233872,-73.8291904199145,dbfs:/Volumes/51810_databricks/sebastian_velasco/landing/landing/derby/nyra_tracking_table.csv,2026-10-17T02:03:43.918Z,1.38,0.25,5.52,19.87,2026-10-17T02:13:40.632Z,315,2,316
AQU,2019-01-01,1,1,6,40.6694478547432,-73.8291836305108,dbfs:/Volumes/51810_databricks/sebastian_velasco/landing/landing/derby/nyra_tracking_table.csv,2026-10-17T02:03:43.918Z,1.798,0.25,7.192,25.89,2026-10-17T02:13:40.632Z,315,2,316


In [0]:
with_quarter = (
    base
    # 1. Posición relativa dentro de la carrera de ESE caballo: 0 = inicio, casi 1 = final
    .withColumn(
        "pos",
        (F.col("trakus_index") - F.col("idx_min"))
        / (F.col("idx_max") - F.col("idx_min") + 1)
    )
    # 2. Convertir 0..1 en cuarto 0, 1, 2 o 3
    .withColumn("quarter", F.least(F.floor(F.col("pos") * 4), F.lit(3)))
    .select(*START_KEYS, "trakus_index", "speed_kmh", "n_frames", "idx_min", "idx_max", "pos", "quarter")
)

with_quarter.limit(5).display()

track_id,race_date,race_number,program_number,trakus_index,speed_kmh,n_frames,idx_min,idx_max,pos,quarter
AQU,2019-01-01,1,1,2,7.62,315,2,316,0.0,0
AQU,2019-01-01,1,1,3,10.58,315,2,316,0.0031746031746031746,0
AQU,2019-01-01,1,1,4,15.98,315,2,316,0.006349206349206349,0
AQU,2019-01-01,1,1,5,19.87,315,2,316,0.009523809523809525,0
AQU,2019-01-01,1,1,6,25.89,315,2,316,0.012698412698412698,0


In [0]:
(with_quarter
 .groupBy("quarter")
 .agg(F.count("*").alias("n_frames"))
 .orderBy("quarter")
 .display())

quarter,n_frames
0,1307074
1,1299848
2,1303489
3,1295825


In [0]:
per_horse = (
    with_quarter
    .groupBy(*START_KEYS)
    .agg(
        F.count("*").alias("n_frames"),
        ((F.max("trakus_index") - F.min("trakus_index")) / F.lit(TRACKING_HZ)).alias("race_duration_s"),

        # Velocidad media en cada cuarto
        F.avg(F.when(F.col("quarter") == 0, F.col("speed_kmh"))).alias("speed_q1"),
        F.avg(F.when(F.col("quarter") == 1, F.col("speed_kmh"))).alias("speed_q2"),
        F.avg(F.when(F.col("quarter") == 2, F.col("speed_kmh"))).alias("speed_q3"),
        F.avg(F.when(F.col("quarter") == 3, F.col("speed_kmh"))).alias("speed_q4"),

        # Regularidad del ritmo
        (F.stddev("speed_kmh") / F.avg("speed_kmh")).alias("speed_cv"),
    )
)

print("Filas:", per_horse.count())
per_horse.limit(5).display()

Filas: 14900


track_id,race_date,race_number,program_number,n_frames,race_duration_s,speed_q1,speed_q2,speed_q3,speed_q4,speed_cv
AQU,2019-01-01,1,1,315,78.5,66.35848101265823,70.74139240506331,67.60721518987344,58.62205128205127,0.15106600514169002
AQU,2019-01-01,2,2,299,74.5,67.45119999999997,68.09333333333336,62.20599999999999,54.54094594594596,0.13681939838354362
AQU,2019-01-01,2,6,297,74.25,68.80826666666667,69.71743243243246,66.9717333333333,58.893972602739716,0.14654853186996158
AQU,2019-01-01,3,2,278,69.25,63.60242857142855,68.97884057971015,63.275,57.01724637681157,0.16603873234068178
AQU,2019-01-01,4,1A,451,112.5,68.3111504424779,68.33008849557518,63.693805309734515,57.18687499999999,0.11970100004083814


In [0]:
per_horse = (
    per_horse
    .withColumn(
        "avg_q",
        (F.col("speed_q1") + F.col("speed_q2") + F.col("speed_q3") + F.col("speed_q4")) / 4
    )
    .withColumn("early_speed_ratio", F.col("speed_q1") / F.col("avg_q"))
    .withColumn("late_speed_ratio",  F.col("speed_q4") / F.col("avg_q"))
    .drop("avg_q")   # era solo auxiliar
)

per_horse.select(
    *START_KEYS, "speed_q1", "speed_q2", "speed_q3", "speed_q4",
    "early_speed_ratio", "late_speed_ratio"
).limit(5).display()

track_id,race_date,race_number,program_number,speed_q1,speed_q2,speed_q3,speed_q4,early_speed_ratio,late_speed_ratio
AQU,2019-01-01,1,1,66.35848101265823,70.74139240506331,67.60721518987344,58.62205128205127,1.0079929785281976,0.8904757188151392
AQU,2019-01-01,2,2,67.45119999999997,68.09333333333336,62.20599999999999,54.54094594594596,1.069417012301608,0.8647290998769043
AQU,2019-01-01,2,6,68.80826666666667,69.71743243243246,66.9717333333333,58.893972602739716,1.0410061046804915,0.8910119085740327
AQU,2019-01-01,3,2,63.60242857142855,68.97884057971015,63.275,57.01724637681157,1.0060749689604964,0.9019093400551773
AQU,2019-01-01,4,1A,68.3111504424779,68.33008849557518,63.693805309734515,57.18687499999999,1.06105376415355,0.8882641938525595


In [0]:
# Ventana ORDENADA: ahora sí importa el orden de los frames
horse_ordered_w = Window.partitionBy(*START_KEYS).orderBy("trakus_index")

with_delta = (
    with_quarter
    .withColumn("prev_speed", F.lag("speed_kmh").over(horse_ordered_w))
    .withColumn("delta_kmh", F.col("speed_kmh") - F.col("prev_speed"))
)

# Distribución del cambio de velocidad entre frames consecutivos (0,25 s)
(with_delta
 .filter(F.col("delta_kmh").isNotNull())
 .select(F.percentile_approx(
     "delta_kmh", [0.0001, 0.001, 0.01, 0.05, 0.5, 0.95, 0.99, 0.999]
 ).alias("percentiles"))
 .display())

with_delta.select("trakus_index", "speed_kmh", "prev_speed", "delta_kmh").limit(5).display()

percentiles
"List(-76.43, -11.97999999999999, -5.170000000000002, -2.9399999999999977, 0.0, 3.3299999999999983, 6.43, 13.95)"


trakus_index,speed_kmh,prev_speed,delta_kmh
2,7.62,null,null
3,10.58,7.62,2.96
4,15.98,10.58,5.4
5,19.87,15.98,3.8900000000000006
6,25.89,19.87,6.02


In [0]:
d = with_delta.filter(F.col("delta_kmh").isNotNull())
total_horses = per_horse.count()

for umbral in [-5, -8, -10, -12, -15]:
    eventos = d.filter(F.col("delta_kmh") < umbral)
    n_frames = eventos.count()
    n_horses = eventos.select(*START_KEYS).distinct().count()
    print(f"delta < {umbral:>4} km/h: {n_frames:>7,} frames | "
          f"{n_horses:>6,} caballos ({n_horses / total_horses:.1%}) | "
          f"{n_frames / total_horses:.2f} eventos por caballo")

# Ver el caso más extremo con sus frames vecinos
peor = d.orderBy("delta_kmh").limit(1).collect()[0]
print("\nPeor frenazo:", peor["delta_kmh"], "en trakus_index", peor["trakus_index"])

(with_delta
 .filter((F.col("track_id") == peor["track_id"]) &
         (F.col("race_date") == peor["race_date"]) &
         (F.col("race_number") == peor["race_number"]) &
         (F.col("program_number") == peor["program_number"]) &
         (F.col("trakus_index").between(peor["trakus_index"] - 4, peor["trakus_index"] + 4)))
 .select("trakus_index", "speed_kmh", "prev_speed", "delta_kmh")
 .orderBy("trakus_index")
 .display())

delta <   -5 km/h:  57,046 frames |  9,926 caballos (66.6%) | 3.83 eventos por caballo
delta <   -8 km/h:  14,865 frames |  4,768 caballos (32.0%) | 1.00 eventos por caballo
delta <  -10 km/h:   8,293 frames |  3,264 caballos (21.9%) | 0.56 eventos por caballo
delta <  -12 km/h:   5,147 frames |  2,324 caballos (15.6%) | 0.35 eventos por caballo
delta <  -15 km/h:   2,890 frames |  1,472 caballos (9.9%) | 0.19 eventos por caballo

Peor frenazo: -76.43 en trakus_index 265


trakus_index,speed_kmh,prev_speed,delta_kmh
261,2.35,2.58,-0.22999999999999998
262,3.92,2.35,1.5699999999999998
263,29.15,3.92,25.229999999999997
264,82.9,29.15,53.75000000000001
265,6.47,82.9,-76.43
266,6.72,6.47,0.25
267,10.9,6.72,4.180000000000001


In [0]:
STOP_KMH = 5   # por debajo de esto consideramos al caballo "quieto"

(with_quarter
 .groupBy("quarter")
 .agg(
     F.count("*").alias("frames"),
     F.sum(F.when(F.col("speed_kmh") < STOP_KMH, 1).otherwise(0)).alias("frames_quietos"),
 )
 .withColumn("pct_quietos", F.round(F.col("frames_quietos") / F.col("frames") * 100, 2))
 .orderBy("quarter")
 .display())

# ¿Cuántos caballos tienen frames quietos al inicio y al final de su registro?
first_last = (with_quarter
    .withColumn("rn_ini", F.row_number().over(horse_ordered_w))
    .withColumn("rn_fin", F.row_number().over(
        Window.partitionBy(*START_KEYS).orderBy(F.col("trakus_index").desc())))
)
print("Caballos con el PRIMER frame quieto:",
      first_last.filter((F.col("rn_ini") == 1) & (F.col("speed_kmh") < STOP_KMH)).count())
print("Caballos con el ÚLTIMO frame quieto:",
      first_last.filter((F.col("rn_fin") == 1) & (F.col("speed_kmh") < STOP_KMH)).count())

quarter,frames,frames_quietos,pct_quietos
0,1307074,31,0.0
1,1299848,0,0.0
2,1303489,70,0.01
3,1295825,1046,0.08


Caballos con el PRIMER frame quieto: 0
Caballos con el ÚLTIMO frame quieto: 56


### Umbral de desaceleración brusca (`n_decel_events`)

Un frenazo se define como una caída de velocidad entre dos frames consecutivos (0,25 s)
menor a `DECEL_KMH`. Se evaluaron varios umbrales:

| Umbral | Frames | Caballos con ≥1 evento | Eventos por caballo |
|---|---|---|---|
| −5 km/h | 57.265 | 66,8% | 3,84 |
| −8 km/h | 14.976 | 32,1% | 1,01 |
| −10 km/h | 8.377 | 22,0% | 0,56 |
| **−12 km/h** | **5.206** | **15,8%** | **0,35** |
| −15 km/h | 2.932 | 10,0% | 0,20 |

Se elige **`DECEL_KMH = −12`**. El umbral de −8 km/h marca a un tercio de los caballos: un
evento tan común no sirve como señal de incidente. Con −12 km/h el frenazo es la excepción
(15,8% de los caballos), corresponde aproximadamente al percentil 0,1% de los cambios entre
frames y equivale a perder 12 km/h en 0,25 s (~13 m/s², más de 1 g), una desaceleración
compatible con un tropiezo, un choque o un caballo sujetado de golpe.

**Limitación conocida:** algunos picos de ruido GPS por debajo de 90 km/h sobreviven a la
limpieza y generan un par subida-caída (p. ej. 0 → 85 → 0). Esos casos inflan
`max_decel_kmh` y se revisan en la inspección de anomalías de la Parte 2.

In [0]:
DECEL_KMH = -12

horse_events = (
    with_delta
    .groupBy(*START_KEYS)
    .agg(
        # Frenazos
        F.sum(F.when(F.col("delta_kmh") < DECEL_KMH, 1).otherwise(0)).alias("n_decel_events"),
        F.min("delta_kmh").alias("max_decel_kmh"),

        # En qué frame alcanzó su velocidad máxima
        F.max_by("trakus_index", "speed_kmh").alias("idx_top_speed"),
        F.min("trakus_index").alias("idx_start"),
        F.max("trakus_index").alias("idx_end"),
    )
    .withColumn(
        "time_to_top_speed_frac",
        (F.col("idx_top_speed") - F.col("idx_start")) / (F.col("idx_end") - F.col("idx_start"))
    )
    .drop("idx_top_speed", "idx_start", "idx_end")
)

dynamics = per_horse.join(horse_events, on=START_KEYS, how="inner")

print("Filas per_horse:", per_horse.count())
print("Filas dynamics:", dynamics.count())
print("Caballos con al menos 1 frenazo:", dynamics.filter(F.col("n_decel_events") > 0).count())

dynamics.select(
    "n_decel_events", "max_decel_kmh", "time_to_top_speed_frac"
).summary("min", "25%", "50%", "75%", "max").display()

Filas per_horse: 14900
Filas dynamics: 14900
Caballos con al menos 1 frenazo: 2324


summary,n_decel_events,max_decel_kmh,time_to_top_speed_frac
min,0,-76.43,0.0
25%,0,-9.260000000000005,0.14921465968586387
50%,0,-6.100000000000001,0.19601328903654486
75%,0,-4.549999999999997,0.29491525423728815
max,31,-1.6700000000000017,1.0


In [0]:
# Misma expresión de start_id que gold_derby_ml_features (separador "||" y casts a string)
start_id_expr = F.sha2(
    F.concat_ws("||",
        F.col("track_id"), F.col("race_date").cast("string"),
        F.col("race_number").cast("string"), F.col("program_number"),
    ), 256
)

race_dynamics = (
    dynamics
    .select(
        # --- Primary key determinista ---
        start_id_expr.alias("start_id"),

        # --- Identificadores ---
        "track_id", "race_date", "race_number", "program_number",

        # --- Duración ---
        "n_frames", "race_duration_s",

        # --- Perfil de velocidad por cuartos ---
        "speed_q1", "speed_q2", "speed_q3", "speed_q4",

        # --- Features de forma ---
        "early_speed_ratio", "late_speed_ratio", "time_to_top_speed_frac", "speed_cv",

        # --- Incidentes ---
        "n_decel_events", "max_decel_kmh",
    )
    .withColumn("_created_at", F.current_timestamp())
)

In [0]:
# 1. PK única y sin nulos
n_rows = race_dynamics.count()
n_distinct = race_dynamics.select("start_id").distinct().count()
n_null_pk = race_dynamics.filter(F.col("start_id").isNull()).count()

print(f"Filas:               {n_rows:,}")
print(f"start_id distintos:  {n_distinct:,}")
print(f"start_id nulos:      {n_null_pk}")

assert n_rows == n_distinct, "La PK start_id NO es única"
assert n_null_pk == 0, "Hay start_id nulos"

# 2. Nulos en las features (KMeans no acepta NULL)
feature_cols = ["speed_q1", "speed_q2", "speed_q3", "speed_q4",
                "early_speed_ratio", "late_speed_ratio",
                "time_to_top_speed_frac", "speed_cv",
                "n_decel_events", "max_decel_kmh"]
race_dynamics.select(
    [F.sum(F.col(c).isNull().cast("int")).alias(c) for c in feature_cols]
).display()

# 3. ¿El start_id cruza con gold_derby_ml_features?
ml = spark.table(T_GOLD_ML_FEATURES).select("start_id")
n_match = race_dynamics.join(ml, on="start_id", how="inner").count()
print(f"Caballos que cruzan con gold_derby_ml_features: {n_match:,} de {n_rows:,}")

# 4. Plausibilidad de las velocidades por cuarto
race_dynamics.select("speed_q1", "speed_q2", "speed_q3", "speed_q4").describe().display()

Filas:               14,900
start_id distintos:  14,900
start_id nulos:      0


speed_q1,speed_q2,speed_q3,speed_q4,early_speed_ratio,late_speed_ratio,time_to_top_speed_frac,speed_cv,n_decel_events,max_decel_kmh
0,0,0,0,0,0,0,0,0,0


Caballos que cruzan con gold_derby_ml_features: 14,899 de 14,900


summary,speed_q1,speed_q2,speed_q3,speed_q4
count,14900,14900,14900,14900
mean,67.33213846067828,71.21128818087625,67.83845151928125,60.352198926980044
stddev,2.745672879908882,3.038325146279679,3.682847894673678,6.488195996808565
min,45.23653846153846,53.0691489361702,6.2904081632653055,0.0
max,78.88866666666667,80.52749999999997,80.61172413793102,74.96428571428571


### Validación de `gold_derby_race_dynamics`

**Integridad:**
- 14.900 filas y 14.900 `start_id` distintos: la PK es única.
- 0 nulos en `start_id` y en las 10 features: la tabla se puede usar directamente en KMeans
  sin imputación.
- `start_id` se calcula con la misma expresión de `gold_derby_ml_features` (`sha2` de las
  `START_KEYS` concatenadas con `"||"`). **14.899 de 14.900** caballos cruzan con esa tabla.
  El único que no cruza es un caballo presente en el tracking pero ausente en
  `silver_derby_starts`, de donde se construye `gold_derby_ml_features`.

**Plausibilidad de las velocidades por cuarto:**

| | Q1 | Q2 | Q3 | Q4 |
|---|---|---|---|---|
| Media (km/h) | 67,3 | 71,2 | 67,8 | 60,4 |
| Desv. estándar | 2,7 | 3,0 | 3,7 | 6,5 |

- El perfil medio sube del Q1 al Q2 (velocidad máxima) y cae en el Q4, consistente con el
  desgaste típico de una carrera. El valor medio de 71 km/h en el cuarto más rápido es
  coherente con la velocidad punta de un caballo de carreras (~60–70 km/h sostenidos).
- La dispersión del Q4 es más del doble que la del Q1: los caballos arrancan de forma
  parecida y se diferencian sobre todo en el cierre. Se espera que `late_speed_ratio`
  sea una feature determinante en el clustering.
- **Valores raros:** el Q3 mínimo (6,3 km/h) y el Q4 mínimo (0 km/h) corresponden a
  caballos que se detuvieron durante la carrera; se conservan como candidatos a anomalía.
  Los máximos cercanos a 80 km/h son altos para el promedio de un cuarto completo y
  también se revisarán en la detección de anomalías.

In [0]:
(race_dynamics
 .write
 .format("delta")
 .mode("overwrite")
 .option("overwriteSchema", "true")
 .saveAsTable(T_GOLD_RACE_DYNAMICS))

print("Tabla escrita:", T_GOLD_RACE_DYNAMICS)

Tabla escrita: `51810_databricks`.`sebastian_velasco`.gold_derby_race_dynamics


In [0]:
spark.table(T_GOLD_RACE_DYNAMICS).display()

start_id,track_id,race_date,race_number,program_number,n_frames,race_duration_s,speed_q1,speed_q2,speed_q3,speed_q4,early_speed_ratio,late_speed_ratio,time_to_top_speed_frac,speed_cv,n_decel_events,max_decel_kmh,_created_at
e97f62b3d72cc0b447bfce00a2763c9e6372d412fb059c17a2f88b19df71000e,AQU,2019-01-01,1,1,315,78.5,66.35848101265823,70.74139240506331,67.60721518987344,58.62205128205127,1.0079929785281976,0.8904757188151392,0.1464968152866242,0.15106600514169002,0,-3.1200000000000045,2026-10-17T16:31:02.356Z
2b14947f6b8e95f771b37f063d6947a4928f5f6e41bc8472273e8ce36f3d0f9e,AQU,2019-01-01,1,2,315,78.5,63.5893670886076,70.58632911392405,66.56050632911392,59.66641025641024,0.9767853925546223,0.9165255235741171,0.24203821656050956,0.15920129832217364,0,-3.739999999999995,2026-10-17T16:31:02.356Z
d27945efc710a60c04e0bb962f03a8a877af7dfd986aa606f6a5d7345d1e62da,AQU,2019-01-01,1,3,315,78.5,65.95392405063292,70.7641772151899,65.84417721518987,57.050641025641035,1.016188626913732,0.8790108155488596,0.1464968152866242,0.1588315990065875,0,-4.059999999999995,2026-10-17T16:31:02.356Z
6fc95a21cc0cfb6f49751346ffe8ae96fb0356deefa6da00b0d966068fdc35cf,AQU,2019-01-01,1,4,315,78.5,65.39860759493669,70.72645569620252,66.0645569620253,58.1770512820513,1.0047154992506977,0.8937710950332994,0.23885350318471338,0.1648030707875712,0,-8.169999999999998,2026-10-17T16:31:02.356Z
a5a81eab8982d95f7d2e04dec243233b6b0c2320de05ed59490f4060ccd26537,AQU,2019-01-01,1,5,315,78.5,64.35810126582281,71.79582278481013,68.40822784810128,61.293974358974374,0.9683147373243084,0.9222127053722109,0.21656050955414013,0.15199866460110945,0,-4.319999999999993,2026-10-17T16:31:02.356Z
212202de498f7d8e42012decb6a838cde0289e1645bfc3a66882b4ab179acc77,AQU,2019-01-01,2,1,299,74.5,68.80346666666668,68.87773333333337,65.96226666666668,57.41918918918918,1.0542061857312304,0.8797763740041444,0.16778523489932887,0.12740016535551368,0,-3.430000000000007,2026-10-17T16:31:02.356Z
7b54c1a9c7d2c5b3360f066fbc267f7d62ff9728787dd77157c2f9df3f034ae4,AQU,2019-01-01,2,2,299,74.5,67.45119999999997,68.09333333333336,62.20599999999999,54.54094594594596,1.069417012301608,0.8647290998769043,0.23825503355704697,0.13681939838354362,0,-6.339999999999996,2026-10-17T16:31:02.356Z
8359535d9475c045a5437b3c37ae7c54c401e48dcf297f799de8e156db33086c,AQU,2019-01-01,2,3,299,74.5,67.54693333333334,68.49986666666666,64.79599999999999,59.560540540540536,1.0375739910727817,0.9148967200943865,0.174496644295302,0.11559730562933233,0,-4.1399999999999935,2026-10-17T16:31:02.356Z
84e55d886f35404208ffe35cfc5637742dceb2f834388750d8e87364b5a0c7a0,AQU,2019-01-01,2,4,299,74.5,67.00106666666666,69.16426666666668,64.11120000000001,56.68405405405408,1.0429781056759106,0.8823774047278091,0.17785234899328858,0.12847346374015856,0,-5.729999999999997,2026-10-17T16:31:02.356Z
c88c27554f7bbf2a0e909a0902b5302a385a6ab66849fdca584e6e1b92722e53,AQU,2019-01-01,2,5,297,74.0,68.6548,69.00797297297298,64.99878378378382,58.76459459459459,1.0504656805772938,0.8991387325381412,0.17229729729729729,0.11732675697851007,0,-9.069999999999993,2026-10-17T16:31:02.356Z


In [0]:
print("Filas en la tabla guardada:", spark.table(T_GOLD_RACE_DYNAMICS).count())

Filas en la tabla guardada: 14900
